## Building an Agentic AI System with SAP Generative AI Hub

Agentic AI systems represent an evolution in how we design and deploy language model-based applications. Rather than simply responding to isolated prompts, these systems are built to reason, plan, and act autonomously navigating multi-step tasks, invoking tools, making decisions, and adapting based on context.

At their core, Agentic AI systems treat the model as more than a text generator. The model is empowered to analyze user inputs, determine whether external actions are required (such as calling APIs or querying data), and execute those actions before synthesizing a final response. This enables a more dynamic and interactive experience, especially valuable in real-world use cases that involve external data sources, business logic, or multi-modal interactions.

### Limitations of Static Prompting 

Even state-of-the-art models like GPT-4o have limitations when used in isolation. For instance, asking for the current time or the weather yields generic or outdated responses. These are not model flaws but consequences of static prompting where models operating without access to real-time data or external APIs.

There are a few common approaches to mitigate these limitations:

* Prompt Engineering: Carefully crafting prompts can improve performance, but it doesn’t give the model access to new or real-time data.

* Retrieval-Augmented Generation (RAG): Augments model outputs with real-time or contextual information by retrieving relevant documents.

* Fine-Tuning: Adapts the model to specific domains or tasks by updating its weights based on domain-specific examples.

While each technique is powerful in its own right, they still operate under a relatively passive paradigm: the model reacts based on given inputs. To take the next step, we need to build systems where the model can actively reason over tasks, decide when to use tools, and dynamically incorporate real-world data into its responses.

In [ ]:
## import env variables
from config import init_env
from config import variables
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()

In [ ]:
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage, AssistantMessage
from gen_ai_hub.orchestration.models.template import Template, TemplateValue
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.service import OrchestrationService
from gen_ai_hub.orchestration.models.response_format import ResponseFormatJsonSchema 
from gen_ai_hub.orchestration.models.llm import LLM
import json
import requests
from datetime import datetime

In [ ]:
# Config the orchestration
llm = LLM(name="gpt-4o", version="latest", parameters={"max_tokens": 256, "temperature": 0.2})

config = OrchestrationConfig(
    template=Template(
        messages=[
            SystemMessage("You are a helpful AI Assistent that answer the best you can all the time."),
            UserMessage("{{?question}}"),
        ]
    ),
    llm=llm    
)

orchestration_service = OrchestrationService(config=config)

In [ ]:
# Create request function
def send_request(config, **kwargs):
    template_values = [TemplateValue(name=key, value=value) for key, value in kwargs.items()]
    answer = orchestration_service.run(config=config, template_values=template_values)
    return answer.module_results.llm.choices[0].message.content 

Now we can have a look at the limitation of static prompting from the questions in the below list: 

the model operates purely on its pre-trained knowledge without access to external tools, APIs, or dynamic environments.

In [ ]:
questions = [
    "Who won the nobel prize of Physics in 2025?",
    "Who won the nobel prize of Physics in 2023?",
    "Which city is the capital of China?",
    "How is the weather there now?",
    "What time is it now?"
]


for q in questions:
    result = send_request(config, question=q)
    print(f"Question: {q}")
    print(f"Answer: {result}")
    print("-" * 40)  # Separator for readabilit


Question: Who won the nobel prize of Physics in 2025?
Answer: I'm sorry, but I don't have information about events or awards beyond October 2023. You might want to check the latest news or the official Nobel Prize website for updates on the 2025 Nobel Prize in Physics.
----------------------------------------
Question: Who won the nobel prize of Physics in 2023?
Answer: The Nobel Prize in Physics for 2023 was awarded to Pierre Agostini, Ferenc Krausz, and Anne L'Huillier for their experimental methods that generate attosecond pulses of light for the study of the electron dynamics in matter.
----------------------------------------
Question: Which city is the capital of China?
Answer: The capital of China is Beijing.
----------------------------------------
Question: How is the weather there now?
Answer: I'm sorry, but I don't have real-time capabilities to provide current weather information. You can check the weather by using a weather website or app, or by searching for the weather i

### Exploring the Core Functions of Our Agentic AI Backend

In this section, we’ll break down three key backend functions that power an intelligent, tool-using AI assistant. Each function addresses a different type of user request checking the time, fetching real-world weather, and retrieving knowledge from an enterprise document store using RAG (Retrieval-Augmented Generation).

#### Get local time 

In [ ]:

def get_time_now():
    """Returns the current local time as a formatted string."""
    return {"time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

Test:

In [ ]:
get_time_now()

{'time': '2025-12-12 06:29:43'}

#### Get weather

In [ ]:

def get_weather(latitude, longitude):
    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]

This function calls the Open-Meteo public API and returns current weather conditions for any given pair of latitude and longitude coordinates. 

To test this function, we need to ask LLM for the latitude and longitude value as its input parameter. 

However, later when agent is ready later, we will see that we no longer need to input the the latitude and longitude mannually.

In [ ]:
result = send_request(config, question = "What is latitude and longitude of China's capital city?")
print(result)

The latitude and longitude of Beijing, the capital city of China, are approximately 39.9042° N (latitude) and 116.4074° E (longitude).


In [ ]:
get_weather(39.9042, 116.4074)

{'time': '2025-12-12T06:15',
 'interval': 900,
 'temperature_2m': -4.2,
 'wind_speed_10m': 3.6}

#### Get retrieved data

This function powers a Retrieval-Augmented Generation (RAG) workflow that answers user questions based on prepared data.

In this example I have added test data (Physics in 2025) from the Nobel Price official websie into my local chroma_db. 

In [ ]:
def get_retriever(query):
    
    from gen_ai_hub.proxy.langchain.openai import ChatOpenAI, OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(deployment_id=variables.EMBEDDING_DEPLOYMENT_ID)  # Deployment ID of text-embedding-3-large
    
    from langchain_chroma import Chroma
    vector_store = Chroma(
        collection_name="NobelPrize",
        embedding_function=embeddings,
        persist_directory="./chroma_db",  # Where to save data locally, remove if not necessary
    )
    
    retriever = vector_store.as_retriever(
        search_type="mmr", search_kwargs={"k": 3, "fetch_k": 5}
    )
    return_value=retriever.invoke(query) 

    result = [
        {
            "source": doc.metadata["source"],
            "start_index": doc.metadata["start_index"],
            "page_content": doc.page_content
        }
        for doc in return_value
    ]

    return json.dumps(result, indent=2, ensure_ascii=False)

Test the retriever

In [ ]:
get_retriever("Who won the nobel prize of Physics in 2025?")

'[\n  {\n    "source": "https://www.nobelprize.org/prizes/physics/2025/press-release/",\n    "start_index": 257,\n    "page_content": "7 October 2025\\nThe Royal Swedish Academy of Sciences has decided to award the Nobel Prize in Physics 2025 to\\nJohn ClarkeUniversity of California, Berkeley, USA\\nMichel H. DevoretYale University, New Haven, CT andUniversity of California, Santa Barbara andGoogle Quantum AI, Santa Barbara, CA, USA\\nJohn M. MartinisUniversity of California, Santa Barbara, USA and Qolab, Los Angeles, CA, USA\\n“for the discovery of macroscopic quantum mechanical tunnelling and energy quantisation in an electric circuit”\\nTheir experiments on a chip revealed quantum physics in action\\nA major question in physics is the maximum size of a system that can demonstrate quantum mechanical effects. This year’s Nobel Prize laureates conducted experiments with an electrical circuit in which they demonstrated both quantum mechanical tunnelling and quantised energy levels in a 

### Equipping Agent with External Capabilities

In agentic AI systems, a language model becomes significantly more capable when equipped with external tools, such as retrieving real-time weather data, fetching the current time, or answering enterprise-specific questions from a knowledge base. 

To enable the model to use these tools effectively, each tool must be registered with a clear definition of its purpose, how it should be called, and what input parameters it expects. This registration can be implemented in various ways, depending on your system’s architecture and flexibility requirements.

To beging, we define a tool of registry, which serves as a catalog of functions the AI agent can call during runtime. 

In [ ]:

class ToolRegistry:
    
    #初始化一个空字典 self.tools。
    #这个字典会用工具名作为 key，value 是一个包含工具信息的字典（见下）
    def __init__(self):
        self.tools = {}


    # #注册工具的方法。
    # 传入：
    # name：工具名称（字符串），作为唯一标识；
    # function：真正的可调用对象（通常是你的 Python 函数）；
    # description：工具的文字描述（告诉 LLM/用户这个工具做什么）；
    # parameters：参数的描述（通常用来告诉 LLM该传哪些字段、类型是什么）。
    # 它会把这些信息存到 self.tools[name] 里，形成一个结构化条目。
    def register(self, name, function, description, parameters):
        self.tools[name] = {
            "function": function,
            "description": description,
            "parameters": parameters
        }

    # 导出工具描述给 prompt/Agent 使用。
    # 返回一个字典：工具名 → {description, parameters}。
    # 这样 LLM 能“看到”有哪些工具、它们的用途以及需要哪些参数，从而在推理中选择调用合适的工具。
    # 这是很关键的一步：给 LLM“观测视角”，不暴露函数本身，只暴露它的用途和接口。
    def get_description_for_prompt(self):
        return {
            name: {
                "description": entry["description"],
                "parameters": entry["parameters"]
            } for name, entry in self.tools.items()
        }


    # 根据工具名取回可执行的函数。
    # 如果找不到，会返回 None（而不是抛异常），这对调用方来说要注意处理。
    def get_callable(self, name):
        return self.tools.get(name, {}).get("function")

Using the above registry tool, now we need to define each function into a tool, with its unique name, a function reference, a clear description, and a schema outlining the required input parameters. 

In [ ]:
registry = ToolRegistry()

registry.register(
    "get_weather",
    get_weather,
    "Retrieves current weather data for a given set of geographic coordinates (latitude, longitude).",
    {
        "latitude": "float - The latitude of the location.",
        "longitude": "float - The longitude of the location."
    }
)

registry.register(
    "get_time_now",
    get_time_now,
    "Returns the current local time in YYYY-MM-DD HH:MM:SS format.",
    {}  # No parameters required
)

registry.register(
    "get_retriever",
    get_retriever,
    "Return the answer to the user question based on retrieved context. Always try to find the answer without this tool first, user this tool only in case you can't find the anwser from your training data. It is possilbe that the answer is not contained in the context either",
    {
        "query": "str - The question that the user asks.",
    }
)

Then generates a structured JSON representation that the agent uses to evaluate which tool to call based on the context and its reasoning.

In [ ]:

tool_prompt = json.dumps(registry.get_description_for_prompt(), indent=2)

In [ ]:
# optional - view the prompt
print(tool_prompt)

{
  "get_weather": {
    "description": "Retrieves current weather data for a given set of geographic coordinates (latitude, longitude).",
    "parameters": {
      "latitude": "float - The latitude of the location.",
      "longitude": "float - The longitude of the location."
    }
  },
  "get_time_now": {
    "description": "Returns the current local time in YYYY-MM-DD HH:MM:SS format.",
    "parameters": {}
  },
  "get_retriever": {
    "description": "Return the answer to the user question based on retrieved context. Always try to find the answer without this tool first, user this tool only in case you can't find the anwser from your training data. It is possilbe that the answer is not contained in the context either",
    "parameters": {
      "query": "str - The question that the user asks."
    }
  }
}


### Create AgentExecutor

The AgentExecutor class is the heart of an agentic AI system. It coordinates LLM-based decision making, tool invocation, and response generation

Details to be checked in AgentExecutor.py


In [ ]:
import AgentExecutor

### Run the Agent 

In [ ]:
agent = AgentExecutor.AgentExecutor(llm=llm, tool_registry=registry, verbose=False)


prompt = """Can you tell me:

    1. What is the capital of China?
    2. How is the weather there now?
    3. What time is it now?
    4. Who won the Nobel Prize in Physics in 2023?
    5. Who won the Nobel Prize in Physics in 2024?
    6. Who won the Nobel Prize in Physics in 2025?
"""
response = agent.run(prompt)
print("\n",response)


 Here are the answers to your questions:

1. The capital of China is Beijing.
2. The current weather in Beijing is -4.2°C with a wind speed of 4.0 m/s.
3. The current time is 2025-12-12 06:30:45.
4. The Nobel Prize in Physics in 2023 was awarded to Pierre Agostini, Ferenc Krausz, and Anne L'Huillier for their experimental methods that generate attosecond pulses of light for the study of electron dynamics in matter.
5. Information about the Nobel Prize in Physics for 2024 is not available.
6. The Nobel Prize in Physics in 2025 was awarded to John Clarke, Michel H. Devoret, and John M. Martinis for their discovery of macroscopic quantum mechanical tunnelling and energy quantisation in an electric circuit.


To see what is exactly happening, change the parameter:  

* verbose=True

This example illustrates a clean and efficient execution of an agentic AI loop in response to a simple but real-world query: asking for the current time.

When the user submits their question, the LLM evaluates the query using the context and instructions previously given by the agent framework. Based on its reasoning, it decides that the user is asking for the time which is a use case that maps directly to the get_time_now tool.

The model then responds with a structured decision object containing:

"decision": "tool" – indicating that a tool should be called,
"reason": "The user asked for time." – a justification for transparency and traceability,
"function": "get_time_now" – the name of the function to call,
"parameters": {} – an empty dictionary, since this function does not require input.

 

In [ ]:
agent = AgentExecutor.AgentExecutor(llm=llm, tool_registry=registry, verbose=True)


prompt = """Can you tell me the answers to questions in the blow list:

    1. What is the capital of China?
    2. How is the weather there now?
    3. What time is it now?
    4. Who won the Nobel Prize in Physics in 2023?
    5. Who won the Nobel Prize in Physics in 2024?
    6. Who won the Nobel Prize in Physics in 2025?
 
"""
response = agent.run(prompt)
print("\n",response)


LLM Reasoning:
{
  "tool_calls": [
    {
      "decision": "no_tool",
      "reason": "The user asked for the capital of China, which is Beijing.",
      "function": "",
      "parameters": {}
    },
    {
      "decision": "tool",
      "reason": "The user asked for the current weather in Beijing, China.",
      "function": "get_weather",
      "parameters": {
        "latitude": 39.9042,
        "longitude": 116.4074
      }
    },
    {
      "decision": "tool",
      "reason": "The user asked for the current time.",
      "function": "get_time_now",
      "parameters": {}
    },
    {
      "decision": "no_tool",
      "reason": "The user asked for the Nobel Prize in Physics in 2023, which is within the training data.",
      "function": "",
      "parameters": {}
    },
    {
      "decision": "tool",
      "reason": "The user asked for the Nobel Prize in Physics in 2024, which is beyond the training data.",
      "function": "get_retriever",
      "parameters": {
        "query"